# Lesson 5 - Hosted Web Search and Evidence

Use the Responses hosted web_search tool for current public hurricane-preparedness guidance appropriate for a fictional small-business insurance client.

Live search is nondeterministic, can change, and may incur API/tool cost. Demo mode uses a fixture dated **2026-07-14**; that fixture is historical demonstration material, never "current."

In [ ]:
import os
from datetime import date

from openai import OpenAI

from harborlight_responses.config import api_key_available, resolve_model
from harborlight_responses.fixtures import FIXTURE_LABEL, web_search_fixture
from harborlight_responses.responses import (
    create_web_evidence_response,
    extract_web_sources,
)

selection = resolve_model()
live_requested = os.getenv("HARBORLIGHT_LIVE") == "1"
live = live_requested and api_key_available()
print("Selected model:", selection.model)
print("Mode:", "Live API" if live else "Demo Fixture")

if live:
    execution_date = date.today().isoformat()
    client = OpenAI()
    response = create_web_evidence_response(client, model=selection.model)
    sources = extract_web_sources(response)
    print("Live execution date:", execution_date)
    print()
    print("Generated interpretation and cited answer:")
    print(response.output_text)
    print()
    print("Retrieved source evidence:")
    for source in sources:
        print("-", source["title"], source["url"])
    if not sources:
        print("No source list was returned; treat the result as insufficient evidence.")
else:
    fixture = web_search_fixture()
    print(FIXTURE_LABEL)
    print("Fixture execution date:", fixture["fixture_date"])
    print()
    print("Retrieved facts (authored fixture):")
    for fact in fixture["retrieved_facts"]:
        print("-", fact)
    print()
    print("Source evidence (dated authored fixture):")
    for source in fixture["sources"]:
        print("-", source["title"], source["url"])
    print()
    print("Generated interpretation (authored fixture):")
    for item in fixture["generated_interpretation"]:
        print("-", item)

## Evidence discipline

Retrieved facts and URLs are evidence; the checklist is generated interpretation. Display citations close to the claim they support. If search returns no sources, conflicting claims, or weak publishers, say that evidence is insufficient and do not fill gaps from model confidence. Record the exact execution date and avoid relative phrases such as "today" in saved material.

File search and native remote MCP are version-two topics, not part of this five-lesson curriculum.